In [4]:
"""
ICPAC Montandon STAC Client

A clean, modular implementation for accessing and analyzing Earth observation data
for ICPAC member countries through the Montandon STAC API.

Requirements:
    pip install pystac-client matplotlib geopandas requests
"""

import os
import json
import logging
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional, Any, Union

import pystac_client
import matplotlib.pyplot as plt
import requests


# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


# ICPAC countries with ISO codes
ICPAC_COUNTRIES = {
    'Djibouti': 'DJI',
    'Eritrea': 'ERI',
    'Ethiopia': 'ETH',
    'Kenya': 'KEN',
    'Somalia': 'SOM',
    'South Sudan': 'SSD',
    'Sudan': 'SDN',
    'Uganda': 'UGA',
    'Burundi': 'BDI',
    'Rwanda': 'RWA',
    'Tanzania': 'TZA'
}

# STAC API endpoint
MONTANDON_STAC_URL = "https://montandon-eoapi-stage.ifrc.org/stac"


class ICPACSTACClient:
    """Client for accessing STAC data for ICPAC countries."""

    def __init__(self, stac_url: str = MONTANDON_STAC_URL):
        """
        Initialize the ICPAC STAC client.
        
        Args:
            stac_url: URL of the STAC API
        """
        self.stac_url = stac_url
        self.client = self._connect()
        
    def _connect(self) -> pystac_client.Client:
        """Connect to the STAC API."""
        try:
            client = pystac_client.Client.open(self.stac_url)
            logger.info(f"Successfully connected to STAC API at {self.stac_url}")
            return client
        except Exception as e:
            logger.error(f"Failed to connect to STAC API: {e}")
            raise

    def list_collections(self) -> List[Any]:
        """List all available collections in the STAC catalog."""
        try:
            collections = list(self.client.get_collections())
            logger.info(f"Found {len(collections)} collections")
            return collections
        except Exception as e:
            logger.error(f"Failed to list collections: {e}")
            raise

    def print_collections(self, collections: Optional[List[Any]] = None) -> None:
        """
        Print information about available collections.
        
        Args:
            collections: List of collections to print. If None, fetches all collections.
        """
        if collections is None:
            collections = self.list_collections()
            
        print("\nAvailable Collections:")
        print("=====================")
        
        for collection in collections:
            print(f"ID: {collection.id}")
            print(f"Title: {collection.title or 'No title'}")
            
            if collection.description:
                desc = collection.description
                if len(desc) > 100:
                    desc = f"{desc[:100]}..."
                print(f"Description: {desc}")
            else:
                print("Description: No description")
                
            print("-" * 50)

    def get_collection(self, collection_id: str) -> Any:
        """
        Get a specific collection by ID.
        
        Args:
            collection_id: ID of the collection
            
        Returns:
            The collection object
        """
        try:
            collection = self.client.get_collection(collection_id)
            return collection
        except Exception as e:
            logger.error(f"Failed to get collection {collection_id}: {e}")
            raise

    def get_country_bbox(self, country_name: str) -> Tuple[float, float, float, float]:
        """
        Get the bounding box for a specific ICPAC country.
        
        Args:
            country_name: Name of the ICPAC country
            
        Returns:
            Tuple of (min_x, min_y, max_x, max_y)
        """
        # Approximate bounding boxes for ICPAC countries
        country_bboxes = {
            'Djibouti': (41.66, 10.9, 43.42, 12.71),
            'Eritrea': (36.32, 12.36, 43.13, 18.03),
            'Ethiopia': (33.99, 3.40, 47.98, 14.89),
            'Kenya': (33.91, -4.67, 41.91, 5.51),
            'Somalia': (40.98, -1.65, 51.13, 11.99),
            'South Sudan': (23.89, 3.49, 35.79, 12.24),
            'Sudan': (21.83, 9.34, 38.61, 22.23),
            'Uganda': (29.57, -1.48, 35.00, 4.23),
            'Burundi': (28.99, -4.45, 30.84, -2.31),
            'Rwanda': (28.85, -2.83, 30.90, -1.05),
            'Tanzania': (29.33, -11.75, 40.45, -0.99),
        }
        
        if country_name not in country_bboxes:
            logger.error(f"Bounding box for {country_name} not found")
            raise ValueError(f"Bounding box for {country_name} not available")
        
        return country_bboxes[country_name]

    def search_items(
        self, 
        collection_id: str, 
        bbox: Optional[Tuple[float, float, float, float]] = None,
        country_name: Optional[str] = None, 
        date_range: Optional[Tuple[str, str]] = None
    ) -> List[Any]:
        """
        Search for items in a specific collection.
        
        Args:
            collection_id: ID of the collection to search
            bbox: Bounding box as (min_x, min_y, max_x, max_y)
            country_name: Name of the country (alternative to bbox)
            date_range: Tuple of (start_date, end_date) as strings in ISO format
            
        Returns:
            List of STAC items
        """
        try:
            # Get bbox from country name if provided
            if bbox is None and country_name is not None:
                bbox = self.get_country_bbox(country_name)
            
            if bbox is None:
                raise ValueError("Either bbox or country_name must be provided")
            
            # Prepare search parameters
            search_params = {
                "collections": [collection_id],
                "bbox": bbox,
            }
            
            # Add date range if provided
            if date_range:
                start_date, end_date = date_range
                search_params["datetime"] = f"{start_date}/{end_date}"
            
            # Execute search
            search = self.client.search(**search_params)
            items = list(search.get_items())
            
            location = country_name or f"bbox {bbox}"
            logger.info(f"Found {len(items)} items for {location} in collection {collection_id}")
            return items
        
        except Exception as e:
            location = country_name or f"bbox {bbox}"
            logger.error(f"Failed to search items for {location}: {e}")
            raise

    def search_items_for_country(
        self, 
        collection_id: str, 
        country_name: str, 
        date_range: Optional[Tuple[str, str]] = None
    ) -> List[Any]:
        """
        Search for items in a specific collection for a specific country.
        
        Args:
            collection_id: ID of the collection to search
            country_name: Name of the country
            date_range: Tuple of (start_date, end_date) as strings in ISO format
            
        Returns:
            List of STAC items
        """
        return self.search_items(
            collection_id=collection_id,
            country_name=country_name,
            date_range=date_range
        )

    def search_items_for_all_icpac(
        self, 
        collection_id: str, 
        date_range: Optional[Tuple[str, str]] = None
    ) -> Dict[str, List[Any]]:
        """
        Search for items in a specific collection for all ICPAC countries.
        
        Args:
            collection_id: ID of the collection to search
            date_range: Tuple of (start_date, end_date) as strings in ISO format
            
        Returns:
            Dictionary mapping country names to lists of items
        """
        results = {}
        
        for country in ICPAC_COUNTRIES.keys():
            try:
                items = self.search_items_for_country(
                    collection_id=collection_id,
                    country_name=country,
                    date_range=date_range
                )
                results[country] = items
                logger.info(f"{country}: Found {len(items)} items")
            except Exception as e:
                logger.error(f"Error searching items for {country}: {e}")
                results[country] = []
        
        return results

    def examine_collection(self, collection_id: str) -> Any:
        """
        Examine the structure and properties of a collection.
        
        Args:
            collection_id: ID of the collection to examine
            
        Returns:
            The collection object
        """
        try:
            collection = self.get_collection(collection_id)
            
            # Print basic information
            print(f"\nCollection: {collection.id}")
            print(f"Title: {collection.title or 'No title'}")
            print(f"Description: {collection.description or 'No description'}")
            
            # Print spatial and temporal extent
            if hasattr(collection, "extent") and collection.extent:
                if hasattr(collection.extent, "spatial") and collection.extent.spatial:
                    print(f"Spatial Extent: {collection.extent.spatial.bboxes}")
                
                if hasattr(collection.extent, "temporal") and collection.extent.temporal:
                    print(f"Temporal Extent: {collection.extent.temporal.intervals}")
            
            # Print license information
            if hasattr(collection, "license") and collection.license:
                print(f"License: {collection.license}")
            
            # Print available item assets
            print("\nItem Asset Information:")
            if hasattr(collection, "item_assets") and collection.item_assets:
                for asset_key, asset in collection.item_assets.items():
                    print(f"  - {asset_key}:")
                    if hasattr(asset, "title") and asset.title:
                        print(f"    Title: {asset.title}")
                    if hasattr(asset, "description") and asset.description:
                        print(f"    Description: {asset.description}")
                    if hasattr(asset, "roles") and asset.roles:
                        print(f"    Roles: {asset.roles}")
                    if hasattr(asset, "type") and asset.type:
                        print(f"    Type: {asset.type}")
            else:
                print("  No item asset information available")
            
            return collection
        
        except Exception as e:
            logger.error(f"Failed to examine collection {collection_id}: {e}")
            raise


class ICPACDataProcessor:
    """Processor for STAC data for ICPAC countries."""
    
    def __init__(self, client: ICPACSTACClient):
        """
        Initialize the ICPAC data processor.
        
        Args:
            client: ICPACSTACClient instance
        """
        self.client = client
    
    def download_file(self, url: str, output_path: str) -> str:
        """
        Download a file from a URL to a specific path.
        
        Args:
            url: URL of the file to download
            output_path: Path where the file will be saved
            
        Returns:
            Path to the downloaded file
        """
        try:
            # Create directory if it doesn't exist
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            
            with requests.get(url, stream=True) as r:
                r.raise_for_status()
                with open(output_path, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
            
            logger.info(f"Downloaded file to {output_path}")
            return output_path
        
        except Exception as e:
            logger.error(f"Failed to download file: {e}")
            raise
    
    def items_to_json(self, items: List[Any]) -> List[Dict[str, Any]]:
        """
        Convert STAC items to JSON-serializable format.
        
        Args:
            items: List of STAC items
            
        Returns:
            List of JSON-serializable dictionaries
        """
        serializable_items = []
        
        for item in items:
            # Convert item to dictionary
            item_dict = {
                "id": item.id,
                "collection": item.collection_id,
                "datetime": item.datetime.isoformat() if hasattr(item, "datetime") and item.datetime else None,
                "bbox": item.bbox if hasattr(item, "bbox") else None,
                "properties": item.properties if hasattr(item, "properties") else {},
                "assets": {key: {"href": asset.href} for key, asset in item.assets.items()} if hasattr(item, "assets") else {}
            }
            
            serializable_items.append(item_dict)
        
        return serializable_items
    
    def save_items_to_json(self, items_dict: Dict[str, List[Any]], output_file: str) -> None:
        """
        Save a dictionary of items to a JSON file.
        
        Args:
            items_dict: Dictionary mapping country names to lists of items
            output_file: Path to the output JSON file
        """
        try:
            # Create directory if it doesn't exist
            os.makedirs(os.path.dirname(output_file), exist_ok=True)
            
            # Convert items to serializable format
            serializable_dict = {}
            
            for country, items in items_dict.items():
                serializable_dict[country] = self.items_to_json(items)
            
            # Save to JSON file
            with open(output_file, "w") as f:
                json.dump(serializable_dict, f, indent=2)
            
            logger.info(f"Saved items to {output_file}")
        
        except Exception as e:
            logger.error(f"Failed to save items to JSON: {e}")
            raise
    
    def generate_items_chart(
        self, 
        items_dict: Dict[str, List[Any]], 
        title: str, 
        output_file: str
    ) -> None:
        """
        Generate a bar chart showing the number of items by country.
        
        Args:
            items_dict: Dictionary mapping country names to lists of items
            title: Title of the chart
            output_file: Path to save the chart
        """
        try:
            # Create directory if it doesn't exist
            os.makedirs(os.path.dirname(output_file), exist_ok=True)
            
            # Prepare data for chart
            country_names = list(items_dict.keys())
            item_counts = [len(items) for items in items_dict.values()]
            
            plt.figure(figsize=(12, 8))
            plt.bar(country_names, item_counts)
            plt.title(title)
            plt.xlabel("Country")
            plt.ylabel("Number of Items")
            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()
            
            # Save the figure
            plt.savefig(output_file)
            logger.info(f"Saved chart to {output_file}")
            
            # Close the figure to free memory
            plt.close()
        
        except Exception as e:
            logger.error(f"Failed to generate chart: {e}")
            raise
    
    def generate_collection_report(
        self, 
        collection_id: str, 
        items_dict: Dict[str, List[Any]], 
        collection: Any,
        date_range: Optional[Tuple[str, str]] = None,
        output_dir: str = "reports"
    ) -> Tuple[str, str, str]:
        """
        Generate a comprehensive report for a collection.
        
        Args:
            collection_id: ID of the collection
            items_dict: Dictionary mapping country names to lists of items
            collection: Collection object
            date_range: Tuple of (start_date, end_date) as strings in ISO format
            output_dir: Directory to save the report
            
        Returns:
            Tuple of (report_file, json_file, chart_file) paths
        """
        try:
            # Create output directory
            os.makedirs(output_dir, exist_ok=True)
            
            # Generate file paths
            report_file = os.path.join(output_dir, f"{collection_id}_report.txt")
            json_file = os.path.join(output_dir, f"{collection_id}_items.json")
            chart_file = os.path.join(output_dir, f"{collection_id}_chart.png")
            
            # Save items to JSON
            self.save_items_to_json(items_dict, json_file)
            
            # Generate chart
            chart_title = f"Number of {collection_id} Items by ICPAC Country"
            self.generate_items_chart(items_dict, chart_title, chart_file)
            
            # Generate summary statistics
            total_items = sum(len(items) for items in items_dict.values())
            countries_with_data = sum(1 for items in items_dict.values() if len(items) > 0)
            
            # Generate a text report
            with open(report_file, "w") as f:
                f.write(f"Report for Collection: {collection_id}\n")
                f.write("=" * 50 + "\n\n")
                
                f.write(f"Collection Title: {collection.title or 'No title'}\n")
                f.write(f"Collection Description: {collection.description or 'No description'}\n\n")
                
                if date_range:
                    f.write(f"Date Range: {date_range[0]} to {date_range[1]}\n\n")
                
                f.write(f"Total Items: {total_items}\n")
                f.write(f"Countries with Data: {countries_with_data}/{len(ICPAC_COUNTRIES)}\n\n")
                
                f.write("Items by Country:\n")
                for country, items in items_dict.items():
                    f.write(f"  {country}: {len(items)} items\n")
                
                f.write("\nReports Saved:\n")
                f.write(f"  - JSON File: {os.path.basename(json_file)}\n")
                f.write(f"  - Chart: {os.path.basename(chart_file)}\n")
            
            logger.info(f"Generated report for collection {collection_id}")
            logger.info(f"Report files: {report_file}, {json_file}, {chart_file}")
            
            return report_file, json_file, chart_file
        
        except Exception as e:
            logger.error(f"Failed to generate report: {e}")
            raise


class ICPACAnalyzer:
    """Analyzer for STAC data for ICPAC countries."""
    
    def __init__(self, client: ICPACSTACClient, processor: ICPACDataProcessor):
        """
        Initialize the ICPAC analyzer.
        
        Args:
            client: ICPACSTACClient instance
            processor: ICPACDataProcessor instance
        """
        self.client = client
        self.processor = processor
    
    def analyze_collection(
        self, 
        collection_id: str, 
        date_range: Optional[Tuple[str, str]] = None,
        output_dir: str = "analysis"
    ) -> Tuple[str, str, str]:
        """
        Analyze a collection for all ICPAC countries.
        
        Args:
            collection_id: ID of the collection
            date_range: Tuple of (start_date, end_date) as strings in ISO format
            output_dir: Directory to save the analysis results
            
        Returns:
            Tuple of (report_file, json_file, chart_file) paths
        """
        collection = self.client.examine_collection(collection_id)
        items_dict = self.client.search_items_for_all_icpac(collection_id, date_range)
        
        # Generate report
        return self.processor.generate_collection_report(
            collection_id=collection_id,
            items_dict=items_dict,
            collection=collection,
            date_range=date_range,
            output_dir=output_dir
        )
    
    def analyze_all_collections(
        self, 
        date_range: Optional[Tuple[str, str]] = None,
        output_dir: str = "icpac_analysis"
    ) -> str:
        """
        Analyze all available collections for ICPAC countries.
        
        Args:
            date_range: Tuple of (start_date, end_date) as strings in ISO format
            output_dir: Directory to save the analysis results
            
        Returns:
            Path to the summary file
        """
        try:
            # Create output directory
            os.makedirs(output_dir, exist_ok=True)
            
            # List all collections
            collections = self.client.list_collections()
            
            # date range or default to last 90 days
            if date_range is None:
                end_date = datetime.now().strftime("%Y-%m-%d")
                start_date = (datetime.now() - timedelta(days=90)).strftime("%Y-%m-%d")
                date_range = (start_date, end_date)
            
            # summary file
            summary_file = os.path.join(output_dir, "collections_summary.txt")
            
            with open(summary_file, "w") as f:
                f.write("ICPAC Collections Summary\n")
                f.write("=" * 50 + "\n\n")
                
                f.write(f"Date Range: {date_range[0]} to {date_range[1]}\n")
                f.write(f"Total Collections: {len(collections)}\n\n")
                
                f.write("Collections:\n")
                
                # Analyze each collection
                for collection in collections:
                    collection_id = collection.id
                    f.write(f"\n{collection_id}:\n")
                    f.write("-" * len(collection_id) + "\n")
                    
                    try:
                        # Search for one item in any ICPAC country to check if collection has data
                        has_data = False
                        for country in ICPAC_COUNTRIES.keys():
                            try:
                                items = self.client.search_items_for_country(
                                    collection_id=collection_id,
                                    country_name=country,
                                    date_range=date_range
                                )
                                if items:
                                    has_data = True
                                    break
                            except Exception:
                                continue
                        
                        if has_data:
                            f.write(f"  Has data for ICPAC countries: Yes\n")
                            
                            # Generate report for this collection
                            collection_dir = os.path.join(output_dir, collection_id)
                            
                            report_file, json_file, chart_file = self.analyze_collection(
                                collection_id=collection_id,
                                date_range=date_range,
                                output_dir=collection_dir
                            )
                            
                            f.write(f"  Report: {os.path.basename(report_file)}\n")
                            f.write(f"  JSON: {os.path.basename(json_file)}\n")
                            f.write(f"  Chart: {os.path.basename(chart_file)}\n")
                        else:
                            f.write(f"  Has data for ICPAC countries: No\n")
                    
                    except Exception as e:
                        f.write(f"  Error analyzing collection: {str(e)}\n")
            
            logger.info(f"Analysis complete. Summary saved to {summary_file}")
            return summary_file
        
        except Exception as e:
            logger.error(f"Failed to analyze ICPAC collections: {e}")
            raise


def get_date_range(days: int = 30) -> Tuple[str, str]:
    """
    Get a date range from now to a specified number of days in the past.
    
    Args:
        days: Number of days in the past
        
    Returns:
        Tuple of (start_date, end_date) as strings in ISO format
    """
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
    return start_date, end_date


def main():
    """Main function demonstrating STAC capabilities for ICPAC countries."""
    try:
        # Initialize client and processor
        client = ICPACSTACClient()
        processor = ICPACDataProcessor(client)
        analyzer = ICPACAnalyzer(client, processor)
        
        # Option 1: List all collections
        collections = client.list_collections()
        client.print_collections(collections)
        
        # Option 2: Analyze one specific collection for one country
        country = 'Kenya'
        collection_id = 'sentinel-s2-l2a'  # Common collection, should be available
        
        # Set date range for the last 30 days
        date_range = get_date_range(30)
        start_date, end_date = date_range
        
        print(f"\nSearching for data for {country} from {start_date} to {end_date}")
        try:
            items = client.search_items_for_country(
                collection_id=collection_id,
                country_name=country,
                date_range=date_range
            )
            
            # Display information about found items
            if items:
                print(f"\nFound {len(items)} items. Details of the first item:")
                item = items[0]
                print(f"ID: {item.id}")
                if hasattr(item, 'datetime') and item.datetime:
                    print(f"Date: {item.datetime.strftime('%Y-%m-%d')}")
                print(f"Available assets: {list(item.assets.keys())}")
            else:
                print(f"No items found for {country} in the specified date range.")
        except Exception as e:
            print(f"Error searching for items: {e}")
        
        print("\nWould you like to run a comprehensive analysis of all collections for ICPAC countries?")
        print("This may take some time depending on the number of collections.")
        run_analysis = input("Run analysis? (y/n): ").lower().strip() == 'y'
        
        if run_analysis:
            summary_file = analyzer.analyze_all_collections()
            print(f"\nAnalysis complete. Summary saved to {summary_file}")
    
    except Exception as e:
        logger.error(f"An error occurred in the main function: {e}")


if __name__ == "__main__":
    main()

2025-04-09 06:17:26,738 - __main__ - INFO - Successfully connected to STAC API at https://montandon-eoapi-stage.ifrc.org/stac
2025-04-09 06:17:27,252 - __main__ - INFO - Found 29 collections
2025-04-09 06:17:27,374 - __main__ - INFO - Found 0 items for Kenya in collection sentinel-s2-l2a



Available Collections:
ID: desinventar-events
Title: DesInventar Mapped Events
Description: Events mapped from the DesInventar disaster loss database. DesInventar is a conceptual and methodolo...
--------------------------------------------------
ID: desinventar-impacts
Title: DesInventar Impacts
Description: Impact records mapped from the DesInventar disaster loss database. DesInventar is a conceptual and m...
--------------------------------------------------
ID: emdat-events
Title: EM-DAT Source Events
Description: Global Disaster Events from the Emergency Events Database (EM-DAT). EM-DAT is a global database on n...
--------------------------------------------------
ID: emdat-hazards
Title: EM-DAT Source Hazards
Description: Hazard records from the Emergency Events Database (EM-DAT). EM-DAT is a global database on natural a...
--------------------------------------------------
ID: emdat-impacts
Title: EM-DAT Source Impacts
Description: Impact records from the Emergency Events Data

Run analysis? (y/n):  Y


2025-04-09 06:17:31,586 - __main__ - INFO - Found 29 collections
2025-04-09 06:17:31,713 - __main__ - INFO - Found 0 items for Djibouti in collection desinventar-events
2025-04-09 06:17:31,826 - __main__ - INFO - Found 0 items for Eritrea in collection desinventar-events
2025-04-09 06:17:31,941 - __main__ - INFO - Found 2 items for Ethiopia in collection desinventar-events
2025-04-09 06:17:32,147 - __main__ - INFO - Found 0 items for Djibouti in collection desinventar-events
2025-04-09 06:17:32,147 - __main__ - INFO - Djibouti: Found 0 items



Collection: desinventar-events
Title: DesInventar Mapped Events
Description: Events mapped from the DesInventar disaster loss database. DesInventar is a conceptual and methodological tool for generating national disaster databases that provides access to disaster effects information at various scales, maintained by the United Nations Office for Disaster Risk Reduction (UNDRR). The database covers primarily Latin American countries and includes detailed information on disaster events with their impacts. Each event includes information on the hazard type, location, date, and various impact metrics. More information on the DesInventar mapping in Monty can be found in the [DesInventar Event Source Mappings](https://github.com/IFRCGo/monty-stac-extension/tree/main/model/sources/DesInventar#event-item).
Spatial Extent: [[-180, -90, 180, 90]]
Temporal Extent: [[datetime.datetime(1800, 1, 1, 0, 0, tzinfo=tzutc()), None]]
License: proprietary

Item Asset Information:
  No item asset informatio

...


Collection: usgs-hazards
Title: USGS Hazards
Description: Hazard records from the United States Geological Survey (USGS) ShakeMap products. The USGS ShakeMap provides near real-time maps of ground motion and shaking intensity following significant earthquakes. These maps include detailed information about ground shaking intensity, peak ground acceleration, peak ground velocity, and potential damage. Each hazard record includes comprehensive data about the earthquake's impact on the surrounding area, with specialized visualizations and data products. More information on the USGS mapping in Monty can be found in the [USGS Hazard Source Mappings](https://github.com/IFRCGo/monty-stac-extension/tree/main/model/sources/USGS#hazard-item-from-shakemap).
Spatial Extent: [[-180, -90, 180, 90]]
Temporal Extent: [[datetime.datetime(2025, 1, 7, 1, 5, 16, tzinfo=tzutc()), datetime.datetime(2025, 1, 7, 1, 5, 16, tzinfo=tzutc())]]
License: Apache-2.0

Item Asset Information:
  No item asset informati

2025-04-09 06:18:14,072 - __main__ - INFO - Found 0 items for Eritrea in collection usgs-hazards
2025-04-09 06:18:14,073 - __main__ - INFO - Eritrea: Found 0 items
2025-04-09 06:18:14,181 - __main__ - INFO - Found 2 items for Ethiopia in collection usgs-hazards
2025-04-09 06:18:14,181 - __main__ - INFO - Ethiopia: Found 2 items
2025-04-09 06:18:14,287 - __main__ - INFO - Found 0 items for Kenya in collection usgs-hazards
2025-04-09 06:18:14,288 - __main__ - INFO - Kenya: Found 0 items
2025-04-09 06:18:14,397 - __main__ - INFO - Found 2 items for Somalia in collection usgs-hazards
2025-04-09 06:18:14,397 - __main__ - INFO - Somalia: Found 2 items
2025-04-09 06:18:14,504 - __main__ - INFO - Found 0 items for South Sudan in collection usgs-hazards
2025-04-09 06:18:14,504 - __main__ - INFO - South Sudan: Found 0 items
2025-04-09 06:18:14,613 - __main__ - INFO - Found 2 items for Sudan in collection usgs-hazards
2025-04-09 06:18:14,614 - __main__ - INFO - Sudan: Found 2 items
2025-04-09 06:


Collection: usgs-impacts
Title: USGS Impacts
Description: Impact records from the United States Geological Survey (USGS) PAGER (Prompt Assessment of Global Earthquakes for Response) products. PAGER provides rapid estimates of the impact of significant earthquakes around the world, including the number of people and settlements exposed to severe shaking, as well as possible fatalities and economic losses. The system combines earthquake parameters, population exposure, vulnerability, and economic data to estimate potential impacts. Each impact record includes detailed information about estimated fatalities or economic losses with confidence intervals. More information on the USGS mapping in Monty can be found in the [USGS Impact Source Mappings](https://github.com/IFRCGo/monty-stac-extension/tree/main/model/sources/USGS#impact-items-from-pager).
Spatial Extent: [[-180, -90, 180, 90]]
Temporal Extent: [[datetime.datetime(2025, 1, 7, 1, 5, 16, tzinfo=tzutc()), datetime.datetime(2025, 1, 7

2025-04-09 06:18:15,660 - __main__ - INFO - Found 0 items for Eritrea in collection usgs-impacts
2025-04-09 06:18:15,660 - __main__ - INFO - Eritrea: Found 0 items
2025-04-09 06:18:15,770 - __main__ - INFO - Found 2 items for Ethiopia in collection usgs-impacts
2025-04-09 06:18:15,770 - __main__ - INFO - Ethiopia: Found 2 items
2025-04-09 06:18:15,885 - __main__ - INFO - Found 0 items for Kenya in collection usgs-impacts
2025-04-09 06:18:15,885 - __main__ - INFO - Kenya: Found 0 items
2025-04-09 06:18:15,994 - __main__ - INFO - Found 2 items for Somalia in collection usgs-impacts
2025-04-09 06:18:15,994 - __main__ - INFO - Somalia: Found 2 items
2025-04-09 06:18:16,100 - __main__ - INFO - Found 0 items for South Sudan in collection usgs-impacts
2025-04-09 06:18:16,101 - __main__ - INFO - South Sudan: Found 0 items
2025-04-09 06:18:16,212 - __main__ - INFO - Found 2 items for Sudan in collection usgs-impacts
2025-04-09 06:18:16,213 - __main__ - INFO - Sudan: Found 2 items
2025-04-09 06:


Analysis complete. Summary saved to icpac_analysis/collections_summary.txt
